# Metorial + LangGraph Example

This notebook demonstrates how to use Metorial tools with LangGraph for graph-based agent workflows.

In [ ]:
# Install dependencies
%pip install metorial langgraph langchain-openai python-dotenv

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

assert os.getenv("METORIAL_API_KEY"), "Set METORIAL_API_KEY"
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY"
assert os.getenv("EXA_DEPLOYMENT_ID"), "Set EXA_DEPLOYMENT_ID"

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

from metorial import Metorial
from metorial.integrations.langgraph import create_langgraph_tools

In [ ]:
metorial = Metorial(api_key=os.getenv("METORIAL_API_KEY"))

In [ ]:
async def run_agent(query: str):
  async with metorial.provider_session(
    provider="openai",
    server_deployments=[os.getenv("EXA_DEPLOYMENT_ID")],
  ) as session:
    tools = create_langgraph_tools(session)

    print(f"Available tools: {[t.name for t in tools]}")

    llm = ChatOpenAI(model="gpt-4o")
    agent = create_react_agent(llm, tools)

    final_response = None
    async for event in agent.astream({"messages": [("user", query)]}):
      if "agent" in event:
        final_response = event["agent"]["messages"][-1].content
        print(f"Agent: {final_response}")

    return final_response

In [ ]:
result = await run_agent("Search for the latest Python 3.13 features")
print("\nFinal result:", result)